In [1]:
import optuna 
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import pandas as pd

url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI',
           'DiabetesPedigreeFunction', 'Age', 'Outcome']

df = pd.read_csv(url, names=columns)
df.head()

c:\Users\husna\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [4]:
df.isnull().sum()

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64

In [5]:
import numpy as np

# replace zero values with mean of respective column 
cols_with_missing_vals = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[cols_with_missing_vals] = df[cols_with_missing_vals].replace(0,np.nan)

# impute the missing values with mean of respective column 
df.fillna(df.mean(), inplace=True)

print(df.isnull().sum())

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


In [6]:
X = df.drop('Outcome', axis=1)
y = df['Outcome']

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.3, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

Training set shape: (537, 8)
Testing set shape: (231, 8)


In [8]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# define objective function 
def objective(trial):
    n_estimators = trial.suggest_int('n_estimators',50,200)
    max_depth = trial.suggest_int('max_depth',3,20)

    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )

    score = cross_val_score(model, X_train, y_train, cv=3,scoring='accuracy').mean()
    return score

In [9]:
study = optuna.create_study(direction='maximize',sampler=optuna.samplers.TPESampler())
study.optimize(objective,n_trials=50)

[I 2026-09-28 23:13:30,055] A new study created in memory with name: no-name-a14f907c-67e7-40ff-baab-88a59f89e412
[I 2026-09-28 23:13:31,515] Trial 0 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 131, 'max_depth': 13}. Best is trial 0 with value: 0.7728119180633147.
[I 2026-09-28 23:13:33,140] Trial 1 finished with value: 0.7709497206703911 and parameters: {'n_estimators': 153, 'max_depth': 20}. Best is trial 0 with value: 0.7728119180633147.
[I 2026-09-28 23:13:36,220] Trial 2 finished with value: 0.7765363128491619 and parameters: {'n_estimators': 154, 'max_depth': 7}. Best is trial 2 with value: 0.7765363128491619.
[I 2026-09-28 23:13:37,643] Trial 3 finished with value: 0.7616387337057727 and parameters: {'n_estimators': 82, 'max_depth': 11}. Best is trial 2 with value: 0.7765363128491619.
[I 2026-09-28 23:13:38,694] Trial 4 finished with value: 0.7616387337057727 and parameters: {'n_estimators': 82, 'max_depth': 11}. Best is trial 2 with value: 0.7765363

In [10]:
# print the best result 
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7802607076350093
Best hyperparameters: {'n_estimators': 123, 'max_depth': 17}


In [11]:
from sklearn.metrics import accuracy_score

best_model = RandomForestClassifier(**study.best_trial.params,random_state=42)
best_model.fit(X_train,y_train)

y_pred = best_model.predict(X_test)

test_accuracy = accuracy_score(y_test,y_pred)
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')


Test Accuracy with best hyperparameters: 0.74


## samplers in optuna

In [12]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Define the objective function
def objective(trial):
    # Suggest values for the hyperparameters
    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    max_depth = trial.suggest_int('max_depth', 3, 20)

    # Create the RandomForestClassifier with suggested hyperparameters
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )

    # Perform 3-fold cross-validation and calculate accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()

    return score  # Return the accuracy score for Optuna to maximize


In [13]:
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.RandomSampler())  # We aim to maximize accuracy
study.optimize(objective, n_trials=50)  # Run 50 trials to find the best hyperparameters

[I 2026-09-28 23:22:29,328] A new study created in memory with name: no-name-9ccb8ca1-7303-4dcf-bfc5-be6a8415baea
[I 2026-09-28 23:22:30,624] Trial 0 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 74, 'max_depth': 8}. Best is trial 0 with value: 0.7728119180633147.
[I 2026-09-28 23:22:31,348] Trial 1 finished with value: 0.7672253258845437 and parameters: {'n_estimators': 53, 'max_depth': 11}. Best is trial 0 with value: 0.7728119180633147.
[I 2026-09-28 23:22:32,655] Trial 2 finished with value: 0.7783985102420857 and parameters: {'n_estimators': 71, 'max_depth': 19}. Best is trial 2 with value: 0.7783985102420857.
[I 2026-09-28 23:22:35,401] Trial 3 finished with value: 0.7709497206703911 and parameters: {'n_estimators': 154, 'max_depth': 12}. Best is trial 2 with value: 0.7783985102420857.
[I 2026-09-28 23:22:36,541] Trial 4 finished with value: 0.7635009310986964 and parameters: {'n_estimators': 88, 'max_depth': 16}. Best is trial 2 with value: 0.778398510

In [14]:

# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7802607076350093
Best hyperparameters: {'n_estimators': 129, 'max_depth': 17}


In [15]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')


Test Accuracy with best hyperparameters: 0.74


In [16]:
search_space = {
    'n_estimators': [50, 100, 150, 200],
    'max_depth': [5, 10, 15, 20]
}

In [17]:
# Create a study and optimize it using GridSampler
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.GridSampler(search_space))
study.optimize(objective)

[I 2026-09-28 23:24:01,281] A new study created in memory with name: no-name-8daa4260-9ba4-494d-9b90-47472f15908e
[I 2026-09-28 23:24:03,155] Trial 0 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 100, 'max_depth': 5}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-09-28 23:24:05,560] Trial 1 finished with value: 0.7672253258845437 and parameters: {'n_estimators': 150, 'max_depth': 10}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-09-28 23:24:06,276] Trial 2 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 50, 'max_depth': 15}. Best is trial 2 with value: 0.7728119180633147.
[I 2026-09-28 23:24:08,017] Trial 3 finished with value: 0.7653631284916201 and parameters: {'n_estimators': 100, 'max_depth': 15}. Best is trial 2 with value: 0.7728119180633147.
[I 2026-09-28 23:24:09,274] Trial 4 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 100, 'max_depth': 20}. Best is trial 2 with value: 0.772811

In [18]:

# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7746741154562384
Best hyperparameters: {'n_estimators': 50, 'max_depth': 5}


In [19]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')


Test Accuracy with best hyperparameters: 0.74


## optuna visualization

In [20]:
from optuna.visualization import plot_optimization_history,plot_parallel_coordinate,plot_slice,plot_contour,plot_param_importances

In [21]:
plot_optimization_history(study).show()

In [22]:
plot_parallel_coordinate(study).show()

In [23]:
plot_slice(study).show()

In [24]:
plot_contour(study).show()

In [25]:
plot_param_importances(study).show()

## optimizing multiple ml models

In [26]:
# Importing the required libraries
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

In [27]:
# Define the objective function for Optuna
def objective(trial):
    # Choose the algorithm to tune
    classifier_name = trial.suggest_categorical('classifier', ['SVM', 'RandomForest', 'GradientBoosting'])

    if classifier_name == 'SVM':
        # SVM hyperparameters
        c = trial.suggest_float('C', 0.1, 100, log=True)
        kernel = trial.suggest_categorical('kernel', ['linear', 'rbf', 'poly', 'sigmoid'])
        gamma = trial.suggest_categorical('gamma', ['scale', 'auto'])

        model = SVC(C=c, kernel=kernel, gamma=gamma, random_state=42)

    elif classifier_name == 'RandomForest':
        # Random Forest hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)
        bootstrap = trial.suggest_categorical('bootstrap', [True, False])

        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            bootstrap=bootstrap,
            random_state=42
        )

    elif classifier_name == 'GradientBoosting':
        # Gradient Boosting hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)

        model = GradientBoostingClassifier(
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            random_state=42
        )

    # Perform cross-validation and return the mean accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()
    return score

In [28]:
# Create a study and optimize it using CmaEsSampler
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)

[I 2026-09-28 23:28:22,532] A new study created in memory with name: no-name-77753c40-6baa-4cce-8350-86756f501ec9
[I 2026-09-28 23:28:22,612] Trial 0 finished with value: 0.7094972067039107 and parameters: {'classifier': 'SVM', 'C': 3.7587178952098568, 'kernel': 'sigmoid', 'gamma': 'auto'}. Best is trial 0 with value: 0.7094972067039107.
[I 2026-09-28 23:28:24,334] Trial 1 finished with value: 0.7783985102420856 and parameters: {'classifier': 'RandomForest', 'n_estimators': 134, 'max_depth': 11, 'min_samples_split': 8, 'min_samples_leaf': 9, 'bootstrap': False}. Best is trial 1 with value: 0.7783985102420856.
[I 2026-09-28 23:28:28,803] Trial 2 finished with value: 0.7635009310986964 and parameters: {'classifier': 'RandomForest', 'n_estimators': 300, 'max_depth': 17, 'min_samples_split': 4, 'min_samples_leaf': 8, 'bootstrap': True}. Best is trial 1 with value: 0.7783985102420856.
[I 2026-09-28 23:28:32,141] Trial 3 finished with value: 0.7541899441340782 and parameters: {'classifier': 

In [29]:
# Retrieve the best trial
best_trial = study.best_trial
print("Best trial parameters:", best_trial.params)
print("Best trial accuracy:", best_trial.value)

Best trial parameters: {'classifier': 'SVM', 'C': 0.11134488780924878, 'kernel': 'linear', 'gamma': 'auto'}
Best trial accuracy: 0.7895716945996275


In [30]:
study.trials_dataframe()

,number,value,datetime_start,datetime_complete,duration,params_C,params_bootstrap,params_classifier,params_gamma,params_kernel,params_learning_rate,params_max_depth,params_min_samples_leaf,params_min_samples_split,params_n_estimators,system_attrs_tpe:relative_params:0,state
0,0,0.709497,2026-09-28 23:28:22.535624,2026-09-28 23:28:22.612313,0 days 00:00:00.076689,3.758718,NaN,SVM,auto,sigmoid,NaN,NaN,NaN,NaN,NaN,NaN,COMPLETE
1,1,0.778399,2026-09-28 23:28:22.615231,2026-09-28 23:28:24.334800,0 days 00:00:01.719569,NaN,False,RandomForest,NaN,NaN,NaN,11.0,9.0,8.0,134.0,NaN,COMPLETE
2,2,0.763501,2026-09-28 23:28:24.344256,2026-09-28 23:28:28.802482,0 days 00:00:04.458226,NaN,True,RandomForest,NaN,NaN,NaN,17.0,8.0,4.0,300.0,NaN,COMPLETE
3,3,0.754190,2026-09-28 23:28:28.805613,2026-09-28 23:28:32.141239,0 days 00:00:03.335626,NaN,NaN,GradientBoosting,NaN,NaN,0.094782,7.0,6.0,6.0,214.0,NaN,COMPLETE
4,4,0.757914,2026-09-28 23:28:32.141239,2026-09-28 23:28:32.332475,0 days 00:00:00.191236,0.307339,NaN,SVM,scale,sigmoid,NaN,NaN,NaN,NaN,NaN,NaN,COMPLETE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,95,0.754190,2026-09-28 23:29:37.628531,2026-09-28 23:29:37.709114,0 days 00:00:00.080583,0.236431,NaN,SVM,scale,rbf,NaN,NaN,NaN,NaN,NaN,"{""classifier"": ""SVM""}",COMPLETE
96,96,0.744879,2026-09-28 23:29:37.710113,2026-09-28 23:29:43.208497,0 days 00:00:05.498384,NaN,NaN,GradientBoosting,NaN,NaN,0.137970,14.0,2.0,9.0,212.0,"{""classifier"": ""GradientBoosting""}",COMPLETE
97,97,0.787709,2026-09-28 23:29:43.209621,2026-09-28 23:29:43.322478,0 days 00:00:00.112857,0.155520,NaN,SVM,auto,linear,NaN,NaN,NaN,NaN,NaN,"{""classifier"": ""SVM""}",COMPLETE
98,98,0.785847,2026-09-28 23:29:43.322478,2026-09-28 23:29:43.434670,0 days 00:00:00.112192,0.202665,NaN,SVM,auto,linear,NaN,NaN,NaN,NaN,NaN,"{""classifier"": ""SVM""}",COMPLETE


In [31]:
study.trials_dataframe()['params_classifier'].value_counts()

params_classifier
SVM                 79
RandomForest        11
GradientBoosting    10
Name: count, dtype: int64

In [32]:
study.trials_dataframe().groupby('params_classifier')['value'].mean()

params_classifier
GradientBoosting    0.753818
RandomForest        0.768410
SVM                 0.775806
Name: value, dtype: float64

In [33]:
# 1. Optimization History
plot_optimization_history(study).show()

In [34]:
# 3. Slice Plot
plot_slice(study).show()

In [35]:
# 5. Hyperparameter Importance
plot_param_importances(study).show()

In [37]:
import optuna
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score
import numpy as np

# Load the Iris dataset
X, y = load_iris(return_X_y=True)

# Split the dataset into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define the objective function for XGBoost
def objective(trial):
    # Hyperparameter search space
    param = {
        'verbosity': 0,
        'objective': 'multi:softprob',
        'num_class': 3,
        'eval_metric': 'mlogloss',  # Ensure that the eval_metric is specified here
        'booster': 'gbtree',
        'lambda': trial.suggest_float('lambda', 1e-8, 1.0, log=True),
        'alpha': trial.suggest_float('alpha', 1e-8, 1.0, log=True),
        'eta': trial.suggest_float('eta', 0.01, 0.3),
        'gamma': trial.suggest_float('gamma', 1e-8, 1.0, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 9),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample': trial.suggest_float('subsample', 0.4, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'n_estimators': 300,
    }

    # Create DMatrix for XGBoost
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dtest = xgb.DMatrix(X_test, label=y_test)

    # Define a pruning callback based on evaluation metrics
    pruning_callback = optuna.integration.XGBoostPruningCallback(trial, "eval-mlogloss")  # Match the metric name in the evals list

    # Train the model
    bst = xgb.train(
        param,
        dtrain,
        num_boost_round=300,
        evals=[(dtrain, "train"), (dtest, "eval")],  # Ensure the eval datasets and names are specified
        early_stopping_rounds=30,
        callbacks=[pruning_callback]
    )

    # Predict on the test set
    preds = bst.predict(dtest)
    best_preds = [int(np.argmax(line)) for line in preds]

    # Return accuracy as the objective value
    accuracy = accuracy_score(y_test, best_preds)
    return accuracy

# Create a study with pruning
study = optuna.create_study(direction='maximize', pruner=optuna.pruners.SuccessiveHalvingPruner())
study.optimize(objective, n_trials=50)

# Output the best trial
print(f"Best trial: {study.best_trial.params}")
print(f"Best accuracy: {study.best_value}")


[I 2026-09-28 23:31:10,179] A new study created in memory with name: no-name-af0ab865-0407-4116-83f9-e0cfd9cdb32d
c:\Users\husna\AppData\Local\Programs\Python\Python312\Lib\site-packages\optuna\integration\xgboost.py:14: FutureWarning: `optuna.integration.xgboost` has been deprecated in v4.9.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v4.9.0. Use `optuna_integration.xgboost` instead.
  optuna_warn(f"{msg} Use `optuna_integration.xgboost` instead.", FutureWarning)


[0]	train-mlogloss:0.80385	eval-mlogloss:0.79235
[1]	train-mlogloss:0.60696	eval-mlogloss:0.60813
[2]	train-mlogloss:0.46935	eval-mlogloss:0.46656
[3]	train-mlogloss:0.37026	eval-mlogloss:0.35909
[4]	train-mlogloss:0.29530	eval-mlogloss:0.27979
[5]	train-mlogloss:0.23737	eval-mlogloss:0.22798
[6]	train-mlogloss:0.19383	eval-mlogloss:0.18129
[7]	train-mlogloss:0.16057	eval-mlogloss:0.14432
[8]	train-mlogloss:0.13501	eval-mlogloss:0.11792
[9]	train-mlogloss:0.11378	eval-mlogloss:0.09837
[10]	train-mlogloss:0.09595	eval-mlogloss:0.08576
[11]	train-mlogloss:0.08241	eval-mlogloss:0.07406
[12]	train-mlogloss:0.07124	eval-mlogloss:0.06242
[13]	train-mlogloss:0.06333	eval-mlogloss:0.05258
[14]	train-mlogloss:0.05700	eval-mlogloss:0.04619
[15]	train-mlogloss:0.05177	eval-mlogloss:0.03831
[16]	train-mlogloss:0.04759	eval-mlogloss:0.03364
[17]	train-mlogloss:0.04356	eval-mlogloss:0.03023
[18]	train-mlogloss:0.04047	eval-mlogloss:0.02608
[19]	train-mlogloss:0.04026	eval-mlogloss:0.02625
[20]	train

[I 2026-09-28 23:31:17,826] Trial 0 finished with value: 1.0 and parameters: {'lambda': 4.1337559910672894e-05, 'alpha': 0.011808853729328387, 'eta': 0.24005561243597381, 'gamma': 0.011928557488665113, 'max_depth': 9, 'min_child_weight': 1, 'subsample': 0.7798325311658101, 'colsample_bytree': 0.7619585418840047}. Best is trial 0 with value: 1.0.


[0]	train-mlogloss:1.05116	eval-mlogloss:1.05203
[1]	train-mlogloss:1.00427	eval-mlogloss:1.00422
[2]	train-mlogloss:0.95916	eval-mlogloss:0.95631
[3]	train-mlogloss:0.91759	eval-mlogloss:0.91221
[4]	train-mlogloss:0.87901	eval-mlogloss:0.87236
[5]	train-mlogloss:0.84210	eval-mlogloss:0.83129
[6]	train-mlogloss:0.81131	eval-mlogloss:0.79936
[7]	train-mlogloss:0.77847	eval-mlogloss:0.76427
[8]	train-mlogloss:0.74736	eval-mlogloss:0.73085
[9]	train-mlogloss:0.71845	eval-mlogloss:0.69991
[10]	train-mlogloss:0.69352	eval-mlogloss:0.67340
[11]	train-mlogloss:0.66739	eval-mlogloss:0.64529
[12]	train-mlogloss:0.64197	eval-mlogloss:0.61848
[13]	train-mlogloss:0.61752	eval-mlogloss:0.59301
[14]	train-mlogloss:0.59469	eval-mlogloss:0.56889
[15]	train-mlogloss:0.57505	eval-mlogloss:0.54822
[16]	train-mlogloss:0.55507	eval-mlogloss:0.52635
[17]	train-mlogloss:0.53578	eval-mlogloss:0.50617
[18]	train-mlogloss:0.52022	eval-mlogloss:0.49092
[19]	train-mlogloss:0.50767	eval-mlogloss:0.47704
[20]	train

[I 2026-09-28 23:31:23,454] Trial 1 finished with value: 1.0 and parameters: {'lambda': 0.00043454356035212333, 'alpha': 0.5883278049083334, 'eta': 0.04160337103547449, 'gamma': 2.7390682544338783e-07, 'max_depth': 8, 'min_child_weight': 7, 'subsample': 0.5601290574228397, 'colsample_bytree': 0.9595150633320536}. Best is trial 0 with value: 1.0.


...

[0]	train-mlogloss:1.04123	eval-mlogloss:1.04144


[I 2026-09-28 23:31:56,580] Trial 48 pruned. Trial was pruned at iteration 1.


[0]	train-mlogloss:1.07088	eval-mlogloss:1.07190
[1]	train-mlogloss:1.03941	eval-mlogloss:1.03906
[2]	train-mlogloss:1.03166	eval-mlogloss:1.03258
[3]	train-mlogloss:1.01627	eval-mlogloss:1.01485


[I 2026-09-28 23:31:56,708] Trial 49 pruned. Trial was pruned at iteration 4.


Best trial: {'lambda': 4.1337559910672894e-05, 'alpha': 0.011808853729328387, 'eta': 0.24005561243597381, 'gamma': 0.011928557488665113, 'max_depth': 9, 'min_child_weight': 1, 'subsample': 0.7798325311658101, 'colsample_bytree': 0.7619585418840047}
Best accuracy: 1.0


In [38]:
from optuna.visualization import plot_intermediate_values

# 1. Plot intermediate values during the trials
plot_intermediate_values(study).show()